In [1]:
import re
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pandas.api.types import CategoricalDtype

df_lululemon_reviews = pd.read_csv('./data_final/lullu_product_review.csv')

In [2]:
df_lululemon_reviews

,collect_date,platform,search_keyword,review_id,product_id,product_name,review_date,year,month,review_body,rating,helpful_count,has_image,purchase_option,user_height,user_weight,user_height_group,user_weight_group,product_url
0,2026-04-17,자사몰,prod11710026,384702382,prod11380182,메탈 벤트 테크 숏슬리브 셔츠,2026-04-16,2026,4,부드러운 옷 더운날 입기좋은 재질 부드러운 재질 땀냄새 나지 않는 재질 가격이 사...,5,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2026-04-17,자사몰,prod11710026,384590413,prod11380182,메탈 벤트 테크 숏슬리브 셔츠,2026-04-15,2026,4,같은옷 3개째 같은 색상의. 옷을 3개 구매 \n\n장점 : 오늘은 뭐입지 라는 고...,5,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2026-04-17,자사몰,prod11710026,384591246,prod11380182,메탈 벤트 테크 숏슬리브 셔츠,2026-04-15,2026,4,3점 두께감이있어서 호불호가있고 색감이화면과는 조금상이합니다. 불만족하였는데 리뷰...,3,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2026-04-17,자사몰,prod11710026,384333663,prod11380182,메탈 벤트 테크 숏슬리브 셔츠,2026-04-12,2026,4,운동할때 입기 편해요 색도 맘에 들고 운동할때 땀 흡수도 잘 되고 무엇보다 편합니다...,5,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2026-04-17,자사몰,prod11710026,383421185,prod11380182,메탈 벤트 테크 숏슬리브 셔츠,2026-04-04,2026,4,너무 좋아요 러닝을 하기위해서 샀는데 너무 좋습니다 굉장히 쾌적하고 러닝후 냄새도 ...,5,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
11780,2026-04-17,자사몰,prod8351133,319047822,prod8351133,스쿠바 풀집 후디,2024-09-16,2024,9,보기보다 날씬해보임 총길이는 엉덩이가 덮이는 적당한 길이이나 복부가 타이트하게 맞아...,3,6,NaN,NaN,NaN,NaN,NaN,NaN,NaN
11781,2026-04-17,자사몰,prod8351133,317301123,prod8351133,스쿠바 풀집 후디,2024-08-23,2024,8,스쿠바 풀집 핏이 예뻐요 컬러도 너무 예쁘구요 대신 사이즈가 좀 작게나와서 55기준...,5,9,NaN,NaN,NaN,NaN,NaN,NaN,NaN
11782,2026-04-17,자사몰,prod8351133,261121367,prod8351133,스쿠바 풀집 후디,2023-10-21,2023,10,2단계 사이즈 업 필수!!! 자신의 티셔츠 사이즈보다 2단계는 높여야 지퍼를 채울 ...,4,8,NaN,NaN,NaN,NaN,NaN,NaN,NaN
11783,2026-04-17,자사몰,prod20002603,376950876,prod20002603,저지 트레이닝 머슬 탱크탑 *워드마크,2026-01-22,2026,1,기본 템으로 굿! 일단 입었을때 몸에 편안하게 착 떨어지고 운동 전에도 후에도 불편...,5,1,NaN,S,NaN,NaN,NaN,NaN,NaN


In [3]:
df_lululemon_reviews.info()

<class 'pandas.DataFrame'>
RangeIndex: 11785 entries, 0 to 11784
Data columns (total 19 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   collect_date       11785 non-null  str    
 1   platform           11785 non-null  str    
 2   search_keyword     11785 non-null  str    
 3   review_id          11785 non-null  int64  
 4   product_id         11785 non-null  str    
 5   product_name       11785 non-null  str    
 6   review_date        11785 non-null  str    
 7   year               11785 non-null  int64  
 8   month              11785 non-null  int64  
 9   review_body        11783 non-null  str    
 10  rating             11785 non-null  int64  
 11  helpful_count      11785 non-null  int64  
 12  has_image          0 non-null      float64
 13  purchase_option    4479 non-null   str    
 14  user_height        0 non-null      float64
 15  user_weight        0 non-null      float64
 16  user_height_group  280 non-null  

In [4]:
df_lululemon_reviews.shape

(11785, 19)

### 컬럼명 변경

In [5]:
df_lululemon_reviews = df_lululemon_reviews.rename(columns={
    'date' : 'review_date',
    'review_text' : 'content',
    'helpful' : 'helpful_count',
    'purchased_size' : 'purchase_option',
    'height_group' : 'user_height_group',
    'weight_group' : 'user_weight_group',
    'review_body' : 'content'
})

### 없는 컬럼 추가

In [6]:
# 없는 컬럼 추가하기
required_columns = [
    'collect_date', 'platform', 'review_id', 'product_id', 'product_name',
    'review_date', 'year', 'month', 'content', 'rating', 'helpful_count',
    'has_image', 'purchase_option', 'user_height', 'user_weight',
    'user_height_group', 'user_weight_group', 'product_url'
]

for col in required_columns:
    if col not in df_lululemon_reviews.columns:
        df_lululemon_reviews[col] = None

# 컬럼 순서 정리
df_lululemon_reviews = df_lululemon_reviews[required_columns]

### 키 값 이름 변경
- 190cm -> 190cm 이상
- 150cm -> 150cm 이하 변경

In [7]:
df_lululemon_reviews['user_height_group'] = df_lululemon_reviews['user_height_group'].replace({
    '190cm' : '190cm 이상',
    '150cm' : '150cm 이하'
})

### 몸무게 값 이름 변경
- 90kg -> 90kg 이상
- 44kg -> 44kg 이하 변경

In [8]:
df_lululemon_reviews['user_height_group'].value_counts()

user_height_group
166-170cm    69
161-165cm    68
171-175cm    42
156-160cm    37
176-180cm    31
181-185cm    14
190cm 이상      8
150cm 이하      7
151-155cm     4
Name: count, dtype: int64

In [9]:
df_lululemon_reviews['user_weight_group'] = df_lululemon_reviews['user_weight_group'].replace({
    '90kg' : '90kg 이상',
    '44kg' : '44kg 이하'
})

### 결측치 % 확인

In [10]:
(df_lululemon_reviews.isna().mean() * 100).sort_values(ascending=False)

product_url          100.000000
user_weight          100.000000
user_height          100.000000
has_image            100.000000
user_height_group     97.624098
user_weight_group     63.869325
purchase_option       61.994060
content                0.016971
helpful_count          0.000000
collect_date           0.000000
platform               0.000000
month                  0.000000
year                   0.000000
review_date            0.000000
product_name           0.000000
product_id             0.000000
review_id              0.000000
rating                 0.000000
dtype: float64

### ```review_id``` 중복 확인

In [11]:
# 중복 확인
print(f"전체 행 수: {len(df_lululemon_reviews):,}")
print(f"review_id 기준 중복 수: {df_lululemon_reviews.duplicated(subset=['review_id']).sum():,}")
print(f"review_id 고유값 수: {df_lululemon_reviews['review_id'].nunique():,}")

전체 행 수: 11,785
review_id 기준 중복 수: 0
review_id 고유값 수: 11,785


### 중복 리뷰 데이터 확인 및 제거

In [12]:
multi_name = df_lululemon_reviews.groupby('product_id')['product_name'].nunique()
multi_url = df_lululemon_reviews.groupby('product_id')['product_url'].nunique()

# product_name 매핑 확인
print(multi_name.gt(1).sum(), "개의 product_id가 여러 product_name에 매핑됨")
display(
    df_lululemon_reviews[
        df_lululemon_reviews['product_id'].isin(multi_name[multi_name > 1].index)
    ][['product_id', 'product_name', 'product_url']]
    .drop_duplicates()
    .sort_values('product_id')
)

# product_url 매핑 확인
print(multi_url.gt(1).sum(), "개의 product_id가 여러 product_url에 매핑됨")
display(
    df_lululemon_reviews[
        df_lululemon_reviews['product_id'].isin(multi_url[multi_url > 1].index)
    ][['product_id', 'product_name', 'product_url']]
    .drop_duplicates()
    .sort_values('product_id')
)

0 개의 product_id가 여러 product_name에 매핑됨


,product_id,product_name,product_url


0 개의 product_id가 여러 product_url에 매핑됨


,product_id,product_name,product_url


In [13]:
# ===============================
# 제품명 매칭 확인
# ===============================
multi_name = df_lululemon_reviews.groupby('product_id')['product_name'].nunique()
multi_url = df_lululemon_reviews.groupby('product_id')['product_url'].nunique()

# product_name 매핑 확인
print(multi_name.gt(1).sum(), "개의 product_id가 여러 product_name에 매핑됨")
display(
    df_lululemon_reviews[df_lululemon_reviews['product_id'].isin(multi_name[multi_name > 1].index)]
    [['product_id', 'product_name', 'product_url']]
    .drop_duplicates()
    .sort_values('product_id')
)

# product_url 매핑 확인
print(multi_url.gt(1).sum(), "개의 product_id가 여러 product_url에 매핑됨")
display(
    df_lululemon_reviews[df_lululemon_reviews['product_id'].isin(multi_url[multi_url > 1].index)]
    [['product_id', 'product_name', 'product_url']]
    .drop_duplicates()
    .sort_values('product_id')
)

0 개의 product_id가 여러 product_name에 매핑됨


,product_id,product_name,product_url


0 개의 product_id가 여러 product_url에 매핑됨


,product_id,product_name,product_url


In [14]:
# ===============================
# 리뷰 내용 중복 확인
# ===============================
# 2-1. product_id 같고 content 중복인 데이터 확인
dup_mask = df_lululemon_reviews.duplicated(subset=['product_id', 'content'], keep=False)
print("product_id는 같고 content가 중복 개수: ", dup_mask.sum())
display(df_lululemon_reviews[dup_mask][['product_id', 'product_name', 'review_date', 'content']].sort_values(['product_id', 'content']).head(20))

# 2-2. 위 경우 중 review_date까지 완전히 동일한 경우 개수
print(
    "review_date까지 완전히 동일한 경우 개수: "
    , df_lululemon_reviews[dup_mask].duplicated(subset=['product_id', 'content', 'review_date'], keep=False).sum()
)

# 2-3. product_id, content, review_date, purchase_option 까지 모두 중복인 경우
dup_mask_full = df_lululemon_reviews.duplicated(
    subset=['product_id', 'content', 'review_date', 'purchase_option'],
    keep=False
)
print("purchase_option까지 완전히 동일한 경우 개수: ", dup_mask_full.sum())
display(
    df_lululemon_reviews[dup_mask_full][['product_id', 'product_name', 'review_date', 'purchase_option', 'content']]
    .sort_values(['product_id', 'review_date'])
    .head(20)
)

product_id는 같고 content가 중복 개수:  5


,product_id,product_name,review_date,content
3658,prod11800799,패스트 앤 프리 숏슬리브 셔츠 *Breathe,2025-02-16,나의 러닝을 더 즐겁게 하는 셔츠 패스트 앤 프리 라인업은 단연 런닝에 집중되어 있...
3659,prod11800799,패스트 앤 프리 숏슬리브 셔츠 *Breathe,2025-02-02,나의 러닝을 더 즐겁게 하는 셔츠 패스트 앤 프리 라인업은 단연 런닝에 집중되어 있...
3660,prod11800799,패스트 앤 프리 숏슬리브 셔츠 *Breathe,2025-01-25,나의 러닝을 더 즐겁게 하는 셔츠 패스트 앤 프리 라인업은 단연 런닝에 집중되어 있...
11048,prod11870440,온 마이 레벨 스몰 토트백 5L,2025-07-22,가방 가방이 조그만한 짐들을 넣기에 굉장히 편합니다! \n최고에요!! 남자가들기에...
11049,prod11870440,온 마이 레벨 스몰 토트백 5L,2025-07-22,가방 가방이 조그만한 짐들을 넣기에 굉장히 편합니다! \n최고에요!! 남자가들기에...


review_date까지 완전히 동일한 경우 개수:  2
purchase_option까지 완전히 동일한 경우 개수:  2


,product_id,product_name,review_date,purchase_option,content
11048,prod11870440,온 마이 레벨 스몰 토트백 5L,2025-07-22,NaN,가방 가방이 조그만한 짐들을 넣기에 굉장히 편합니다! \n최고에요!! 남자가들기에...
11049,prod11870440,온 마이 레벨 스몰 토트백 5L,2025-07-22,NaN,가방 가방이 조그만한 짐들을 넣기에 굉장히 편합니다! \n최고에요!! 남자가들기에...


In [15]:
# ===============================
# 중복리뷰 제거
# ===============================
before = len(df_lululemon_reviews)

df_lululemon_reviews = (
    df_lululemon_reviews
    .sort_values(
        by=['helpful_count', 'has_image', 'user_height', 'user_weight', 'user_height_group', 'user_weight_group'],
        ascending=[False, False, False, False, False, False],
        na_position='last'
    )
    .drop_duplicates(
        subset=['product_id', 'content', 'review_date', 'purchase_option'],
        keep='first'
    )
    .sort_index()
)

after = len(df_lululemon_reviews)

print(f"제거 전: {before}개")
print(f"제거된 데이터: {before - after}개")
print(f"남은 데이터: {after}개")

제거 전: 11785개
제거된 데이터: 1개
남은 데이터: 11784개


### ```content``` 텍스트 정제

In [16]:
# HTML 태그 제거
HTML_PATTERN = re.compile(r'<[^>]+>')
# URL 제거
URL_PATTERN = re.compile(r'http\S+|www\.\S+')
# 이메일 제거
EMAIL_PATTERN = re.compile(r'\S+@\S+')
# 특수문자 제거
SPECIAL_PATTERN = re.compile(r'[^\w\s\uAC00-\uD7A3\u3130-\u318F!?]')
# 공백(줄바꿈, 탭) 정리
WHITESPACE_PATTERN = re.compile(r'\s+')
# 반복 문자 축소
REPEAT_PATTERN = re.compile(r'(.)\1{2,}')

def clean_text(text):
    if pd.isna(text):
        return np.nan
    
    text = str(text)

    text = HTML_PATTERN.sub(' ', text)
    text = URL_PATTERN.sub(' ', text)
    text = EMAIL_PATTERN.sub(' ', text)
    text = SPECIAL_PATTERN.sub(' ', text)
    text = REPEAT_PATTERN.sub(r'\1\1', text)
    text = WHITESPACE_PATTERN.sub(' ', text)
# 앞뒤 공백 제거, 소문자화
    text = text.strip().lower()

    return text

df_lululemon_reviews['content'] = df_lululemon_reviews['content'].apply(clean_text)

### ```content``` 5글자 이하 제거

In [17]:
# =========================================
# 띄어쓰기 제거 후 5글자 이하인 데이터 확인
# =========================================
short_mask = df_lululemon_reviews['content'].str.replace(' ', '', regex=False).str.len() <= 5

print(f"5글자 이하 행 수: {short_mask.sum()}")
display(df_lululemon_reviews[short_mask]['content'].value_counts())

# =========================
# 5글자 이하 행 제거
# =========================
before = len(df_lululemon_reviews)
df_lululemon_reviews = df_lululemon_reviews[~short_mask]
after = len(df_lululemon_reviews)

print(f"제거 전: {before}개")
print(f"제거된 행: {before - after}개")
print(f"제거 후: {after}개")

5글자 이하 행 수: 0


Series([], Name: count, dtype: int64)

제거 전: 11784개
제거된 행: 0개
제거 후: 11784개


### ```purchase_option``` 텍스트 정제

In [18]:
df_lululemon_reviews['purchase_option'].value_counts()

purchase_option
S      1551
M      1372
L       650
XS      526
XL      258
2XL     103
XXS      11
3XL       8
Name: count, dtype: int64

In [19]:
# 정규식(패턴) 미리 컴파일 (반복 apply에서 성능 개선)
COLON_PATTERN = re.compile(r':\s+')
SLASH_PATTERN = re.compile(r'\s*/\s*')

def normalize_purchase_option(text):
    if pd.isna(text):
        return np.nan
    # 소문자 통일
    text = str(text).lower()

    # 콜론 뒤 공백 제거 (color: black → color:black)
    text = COLON_PATTERN.sub(':', text)

    # 슬래시 공백 통일 (a/b → a / b)
    text = SLASH_PATTERN.sub(' / ', text)

    # 앞뒤 공백 제거
    text = text.strip()

    return text


df_lululemon_reviews['purchase_option'] = (
    df_lululemon_reviews['purchase_option']
    .apply(normalize_purchase_option)
)

#### 사이즈 자동 정규화

In [20]:
# 사이즈 매핑
size_mapping = {
    'xxs': 90,
    'xs': 95,
    's': 95,
    'm': 100,
    'l': 105,
    'xl': 110,
    '2xl': 115,
    '3xl': 120
}

# 패턴 미리 컴파일 (성능 + 확장성)
SIZE_PATTERN = re.compile(
    r'size\s*[:=]?\s*(xxs|xs|s|m|l|xl|2xl|3xl)',
    re.IGNORECASE
)

def replace_size(text):
    if pd.isna(text):
        return np.nan
    
    text = str(text).lower()

    # size 값만 숫자로 치환
    return SIZE_PATTERN.sub(
        lambda x: f"size:{size_mapping[x.group(1)]}",
        text
    )


# 적용 (원본 컬럼 덮어쓰기)
df_lululemon_reviews['purchase_option'] = (
    df_lululemon_reviews['purchase_option']
    .apply(replace_size)
)

### 파생컬럼 생성

#### ```women_size_yn```

In [21]:
df_lululemon_reviews['women_size_yn'] = (
    df_lululemon_reviews['purchase_option']
    .str.contains(r'\bw\b', case=False, na=False)
    .astype(int)
)

#### ```purchase_option_size```

In [22]:
# size 숫자만 추출
if 'purchase_option' in df_lululemon_reviews.columns:
    df_lululemon_reviews['purchase_option_size'] = (
        df_lululemon_reviews['purchase_option']
        .str.extract(r'size:(\d+)', expand=False)
        .astype('Int16')
    )

df_lululemon_reviews['purchase_option_size'] = df_lululemon_reviews['purchase_option'].map(size_mapping)
df_lululemon_reviews['purchase_option'] = df_lululemon_reviews['purchase_option'].map(size_mapping)

In [23]:
df_lululemon_reviews['purchase_option_size'].value_counts()

purchase_option_size
95.0     2077
100.0    1372
105.0     650
110.0     258
115.0     103
90.0       11
120.0       8
Name: count, dtype: int64

#### ```purchase_option_color```

In [24]:
# 색상 리스트 (필요 시 계속 추가)
COLOR_LIST = [
    'black', 'white', 'brown', 'beige', 'navy', 'gray', 'grey',
    'red', 'blue', 'green', 'pink', 'yellow', 'purple',
    '브라운', '블랙', '화이트', '베이지', '네이비', '그레이', '레드', '블루'
]

COLOR_PATTERN = re.compile(
    r'color:([a-z가-힣]+)|\b(' + '|'.join(COLOR_LIST) + r')\b',
    re.IGNORECASE
)

def extract_color(text):
    if pd.isna(text):
        return np.nan
    
    text = str(text).lower()

    match = COLOR_PATTERN.search(text)
    if match:
        return match.group(1) or match.group(2)
    
    return np.nan


df_lululemon_reviews['purchase_option_color'] = (
    df_lululemon_reviews['purchase_option']
    .apply(extract_color)
)

### 형 변환

- ```month``` 컬럼의 모든 값이 1~12인지 검사

In [25]:
assert df_lululemon_reviews['month'].between(1, 12).all()

In [26]:
# 형 변환(datetime)
date_cols = ['collect_date', 'review_date']

existing_cols = [col for col in date_cols if col in df_lululemon_reviews.columns]

df_lululemon_reviews[existing_cols] = (
    df_lululemon_reviews[existing_cols]
    .apply(pd.to_datetime, errors='coerce')
)

# 형 변환(int32)
df_lululemon_reviews['helpful_count'] = df_lululemon_reviews['helpful_count'].astype('Int32')

# 형 변환(int16)
df_lululemon_reviews['year'] = df_lululemon_reviews['year'].astype('Int16')

# 형 변환(int8)
int8_cols = ['month', 'rating', 'has_image', 'women_size_yn']

existing_cols = [col for col in int8_cols if col in df_lululemon_reviews.columns]

df_lululemon_reviews[existing_cols] = df_lululemon_reviews[existing_cols].astype('Int8')

# 형 변환(float32)
num_cols = ['user_height', 'user_weight']

existing_cols = [col for col in num_cols if col in df_lululemon_reviews.columns]

df_lululemon_reviews[existing_cols] = (
    df_lululemon_reviews[existing_cols]
    .apply(pd.to_numeric, errors='coerce')
    .astype('float32')
)

# 형 변환(category)
cat_cols = [
    'platform',
    'purchase_option_color',
    'purchase_option_size',
    'user_height_group',
    'user_weight_group'
]

existing_cols = [col for col in cat_cols if col in df_lululemon_reviews.columns]

df_lululemon_reviews[existing_cols] = df_lululemon_reviews[existing_cols].astype('category')

# 형 변환(str)
str_cols = ['review_id', 'product_url']

existing_cols = [col for col in str_cols if col in df_lululemon_reviews.columns]

df_lululemon_reviews[existing_cols] = df_lululemon_reviews[existing_cols].astype(str)

In [27]:
# ======================
# 컬럼 순서 재정렬
# ======================

column_order = [
    'collect_date',
    'platform',
    'review_id',
    'product_id',
    'product_name',
    'review_date',
    'year',
    'month',
    'content',
    'rating',
    'helpful_count',
    'has_image',
    'purchase_option',
    'purchase_option_color',
    'purchase_option_size',
    'women_size_yn',
    'user_height',
    'user_weight',
    'user_height_group',
    'user_weight_group',
    'product_url'
]

# 컬럼 존재하는 것만 선택
df_lululemon_reviews = df_lululemon_reviews[
    [col for col in column_order if col in df_lululemon_reviews.columns]
]

# 확인
df_lululemon_reviews.head()

,collect_date,platform,review_id,product_id,product_name,review_date,year,month,content,rating,...,has_image,purchase_option,purchase_option_color,purchase_option_size,women_size_yn,user_height,user_weight,user_height_group,user_weight_group,product_url
0,2026-04-17,자사몰,384702382,prod11380182,메탈 벤트 테크 숏슬리브 셔츠,2026-04-16,2026,4,부드러운 옷 더운날 입기좋은 재질 부드러운 재질 땀냄새 나지 않는 재질 가격이 사악...,5,...,<NA>,NaN,NaN,NaN,0,NaN,NaN,NaN,NaN,NaN
1,2026-04-17,자사몰,384590413,prod11380182,메탈 벤트 테크 숏슬리브 셔츠,2026-04-15,2026,4,같은옷 3개째 같은 색상의 옷을 3개 구매 장점 오늘은 뭐입지 라는 고민 필요없음 ...,5,...,<NA>,NaN,NaN,NaN,0,NaN,NaN,NaN,NaN,NaN
2,2026-04-17,자사몰,384591246,prod11380182,메탈 벤트 테크 숏슬리브 셔츠,2026-04-15,2026,4,3점 두께감이있어서 호불호가있고 색감이화면과는 조금상이합니다 불만족하였는데 리뷰는 ...,3,...,<NA>,NaN,NaN,NaN,0,NaN,NaN,NaN,NaN,NaN
3,2026-04-17,자사몰,384333663,prod11380182,메탈 벤트 테크 숏슬리브 셔츠,2026-04-12,2026,4,운동할때 입기 편해요 색도 맘에 들고 운동할때 땀 흡수도 잘 되고 무엇보다 편합니다...,5,...,<NA>,NaN,NaN,NaN,0,NaN,NaN,NaN,NaN,NaN
4,2026-04-17,자사몰,383421185,prod11380182,메탈 벤트 테크 숏슬리브 셔츠,2026-04-04,2026,4,너무 좋아요 러닝을 하기위해서 샀는데 너무 좋습니다 굉장히 쾌적하고 러닝후 냄새도 ...,5,...,<NA>,NaN,NaN,NaN,0,NaN,NaN,NaN,NaN,NaN


In [ ]:
# 데이터 분리
lululemon_musinsa_reviews_final_s2024 = df_lululemon_reviews[
    df_lululemon_reviews['review_date'] >= '2024-01-01'
].copy()

# 전체 데이터 복사
lululemon_musinsa_reviews_final = df_lululemon_reviews.copy()

# 삭제할 컬럼 리스트
cols_to_drop = ['review_type', 'author', 'comment_count', 'image_urls', 'survey_size', 'survey_width', 'survey_comfort', 'survey_instep']

# 컬럼 제거
lululemon_musinsa_reviews_final_s2024 = lululemon_musinsa_reviews_final_s2024.drop(columns=cols_to_drop, errors='ignore')
lululemon_musinsa_reviews_final = lululemon_musinsa_reviews_final.drop(columns=cols_to_drop, errors='ignore')

# CSV 저장
lululemon_musinsa_reviews_final_s2024.to_csv('data_pre/lululemon_musinsa_reviews_final_s2024.csv', index=False, encoding='utf-8-sig')
lululemon_musinsa_reviews_final.to_csv('data_pre/lululemon_musinsa_reviews_final.csv', index=False, encoding='utf-8-sig')